**Machine Learning Lab**

# 5. Classification II: Classification by Extracting Features

This notebook builds a **news article classifier**: given the text of a BBC news article, predict which category it belongs to (business, politics, sport, tech, or entertainment). Unlike the previous notebook, the raw input here is text, not ready-made numeric columns — so most of the work is turning text into numeric **features** the model can use (Bag of Words, TF-IDF, and hand-picked keyword counts), before comparing several classifiers on the result.

Import the libraries we'll use throughout: pandas/numpy for data handling, matplotlib/seaborn for plots, and several scikit-learn modules for feature extraction, models, and evaluation.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.metrics import classification_report, confusion_matrix

## Loading the Dataset

This notebook uses `bbc_dataset.xlsx`, a spreadsheet of BBC news articles with their text and category label. Run the cell below and upload `bbc_dataset.xlsx` (found in this notebook's folder in the repo) when prompted.

In [ ]:
from google.colab import files
uploaded = files.upload()  # select bbc_dataset.xlsx

Load the spreadsheet into a DataFrame (this needs the `openpyxl` package, which is installed automatically alongside pandas in most environments) and preview it.

In [ ]:
# Load Excel dataset
df = pd.read_excel("bbc_dataset.xlsx")

# Display first few rows
df.head()

Check the dataset size, the set of category labels, and how balanced the classes are.

In [ ]:
print("Number of samples:", len(df))
print("Classes:", df['label'].unique())

# Class distribution
sns.countplot(x="label", data=df)
plt.title("Class Distribution")
plt.show()

## Cleaning the Text

Before extracting features, we normalize the text: lowercase everything, strip out anything that isn't a letter (punctuation, digits), and remove common English **stopwords** ("the", "and", "of", etc.) that carry little topical meaning. `nltk` provides a ready-made stopword list, which we download once with `nltk.download`.

In [ ]:
import re
from nltk.corpus import stopwords
import nltk
nltk.download("stopwords")

stop_words = set(stopwords.words("english"))

def clean_text(text):
    text = text.lower()
    text = re.sub(r"[^a-z\s]", "", text)
    return " ".join([w for w in text.split() if w not in stop_words])

df["clean_text"] = df["text"].apply(clean_text)

Check the new `clean_text` column against the original `text`.

In [ ]:
df.head()

## Train/Test Split

`stratify=df["label"]` ensures the train and test sets each keep roughly the same proportion of every category as the full dataset — important here since the classes aren't perfectly balanced.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(df["clean_text"], df["label"], test_size=0.2, random_state=42, stratify=df["label"])

## Feature Extraction

**Keyword Feature Extraction:** this is a form of manual feature engineering in text classification where we select a set of domain-specific words and count their frequency in each document. Unlike automatic methods such as Bag of Words or TF-IDF, which consider every word in the corpus, keyword features focus on words that are known to be strongly associated with particular categories. For example, "football" or "tournament" are likely indicators of a sports article, while "election" or "parliament" point to politics. Explicitly counting these keywords gives the model a strong, interpretable signal, and combining them with automated features like BoW and TF-IDF creates a hybrid representation that leverages both general vocabulary patterns and domain knowledge.

Define a keyword list spanning all five categories, and a function that counts how many times each keyword appears in a document.

In [ ]:
# Expanded keyword list across categories
keywords = [
    # Business
    "market", "stock", "bank", "economy", "finance", "trade", "growth",
    # Politics
    "election", "government", "minister", "policy", "parliament", "law",
    # Sport
    "football", "cricket", "tennis", "match", "team", "goal", "tournament",
    # Tech
    "technology", "computer", "internet", "software", "ai", "digital", "innovation",
    # Entertainment
    "film", "music", "movie", "actor", "actress", "award", "theatre"
]

# Function to extract keyword counts
def extract_keyword_features(texts, keywords):
    features = []
    for text in texts:
        words = text.split()
        counts = [words.count(kw) for kw in keywords]
        features.append(counts)
    return np.array(features)

# Generate keyword features
X_train_keywords = extract_keyword_features(X_train, keywords)
X_test_keywords = extract_keyword_features(X_test, keywords)

print("Keyword feature shape:", X_train_keywords.shape)

Each row is one document; each column is the count of one keyword in that document.

In [ ]:
X_train_keywords

Now look at keyword usage from a different angle: total frequency of each keyword, broken down by category, across the *whole* dataset.

In [ ]:
from collections import Counter

def keyword_frequency_by_label(df, keywords):
    # Initialize counts
    label_keyword_counts = {label: Counter() for label in df["label"].unique()}

    # Count keyword occurrences per label
    for _, row in df.iterrows():
        words = row["clean_text"].split()
        counts = Counter(words)
        for kw in keywords:
            if kw in counts:
                label_keyword_counts[row["label"]][kw] += counts[kw]

    # Convert to DataFrame
    freq_df = pd.DataFrame(label_keyword_counts).fillna(0).astype(int)
    return freq_df

# Run on your dataset
freq_df = keyword_frequency_by_label(df, keywords)

# Show top keyword frequencies per label
print(freq_df.sort_index())

Visualize the same table as a heatmap, then show the top 10 keywords for each category as separate bar charts.

In [ ]:
# -------------------------------
# Heatmap of keyword frequencies
# -------------------------------
plt.figure(figsize=(12,8))
sns.heatmap(freq_df, annot=True, fmt="d", cmap="YlGnBu")
plt.title("Keyword Frequency per Label")
plt.xlabel("News Label")
plt.ylabel("Keyword")
plt.show()

# -------------------------------
# Bar chart: Top keywords per label
# -------------------------------
for label in df["label"].unique():
    top_keywords = freq_df[label].sort_values(ascending=False).head(10)
    plt.figure(figsize=(8,4))
    sns.barplot(x=top_keywords.values, y=top_keywords.index, hue=top_keywords.index, palette="viridis", legend=False)
    plt.title(f"Top Keywords in {label} Articles")
    plt.xlabel("Frequency")
    plt.ylabel("Keyword")
    plt.show()

Now build the two automatic feature sets: **Bag of Words** (raw word counts) and **TF-IDF** (word counts weighted down for words that appear in many documents, so common-but-uninformative words matter less). `max_features=5000` keeps only the 5000 most frequent words, to keep the feature space manageable. Finally, `hstack` glues the BoW, TF-IDF, and keyword features together into one combined feature matrix per document.

In [ ]:
from scipy.sparse import hstack

# Define vectorizers
bow_vectorizer = CountVectorizer(max_features=5000)
tfidf_vectorizer = TfidfVectorizer(max_features=5000)

# Fit and transform BoW + TF-IDF
X_train_bow = bow_vectorizer.fit_transform(X_train)
X_test_bow = bow_vectorizer.transform(X_test)

X_train_tfidf = tfidf_vectorizer.fit_transform(X_train)
X_test_tfidf = tfidf_vectorizer.transform(X_test)

# Combine (BoW + TF-IDF + Keywords)
X_train_combined = hstack([X_train_bow, X_train_tfidf, X_train_keywords])
X_test_combined = hstack([X_test_bow, X_test_tfidf, X_test_keywords])

print("Final feature shape:", X_train_combined.shape)

## Training and Comparing Multiple Models

With features ready, train five different classifiers on the exact same data and print a `classification_report` plus a confusion matrix for each, so we can compare them directly.

In [ ]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

models = {
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "Naive Bayes": MultinomialNB(),
    "Linear SVM": LinearSVC(),
    "Decision Tree": DecisionTreeClassifier(),
    "Random Forest": RandomForestClassifier(n_estimators=100)
}

results = {}

for name, model in models.items():
    print(f"\n Training {name}...")
    model.fit(X_train_combined, y_train)
    preds = model.predict(X_test_combined)

    print(f"--- {name} Report ---")
    print(classification_report(y_test, preds))

    # Store results
    results[name] = classification_report(y_test, preds, output_dict=True)

    # Confusion Matrix
    cm = confusion_matrix(y_test, preds, labels=model.classes_)
    sns.heatmap(cm, annot=True, fmt="d", xticklabels=model.classes_, yticklabels=model.classes_)
    plt.title(f"Confusion Matrix - {name}")
    plt.show()

Plot overall accuracy side by side to see which model came out on top.

In [ ]:
accuracy_scores = {name: results[name]["accuracy"] for name in results}

plt.figure(figsize=(8,5))
sns.barplot(x=list(accuracy_scores.keys()), y=list(accuracy_scores.values()), hue=list(accuracy_scores.keys()), legend=False)
plt.title("Model Accuracy Comparison")
plt.ylabel("Accuracy")
plt.xticks(rotation=30)
plt.show()

## Using One Model for New Predictions

To classify brand-new articles later on, we need one trained model kept handy (rather than whichever model happened to be last in the loop above). We re-fit a standalone Logistic Regression model here — using the exact same `X_train_combined`/`y_train` as before — so `lr_model` is available on its own for the prediction function below.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

# Initialize and train Logistic Regression model
lr_model = LogisticRegression(max_iter=1000)
print("\nTraining Logistic Regression...")
lr_model.fit(X_train_combined, y_train)
lr_preds = lr_model.predict(X_test_combined)

print("--- Logistic Regression Report ---")
print(classification_report(y_test, lr_preds))

Wrap the full pipeline — cleaning, feature extraction, prediction — into one function that takes raw article text and returns a predicted category. Try it on a brand-new article that wasn't in the dataset at all.

In [ ]:
# --------------------------------
# Predict Category for New Article
# --------------------------------

def predict_news_category(article, model, bow_vectorizer, tfidf_vectorizer, keywords):
    import re
    import numpy as np

    # Preprocess text
    stop_words = set(stopwords.words("english"))
    def clean_text(text):
        text = text.lower()
        text = re.sub(r"[^a-z\s]", "", text)
        return " ".join([w for w in text.split() if w not in stop_words])

    clean_article = clean_text(article)

    # Feature extraction
    # BoW
    bow_feat = bow_vectorizer.transform([clean_article])
    # TF-IDF
    tfidf_feat = tfidf_vectorizer.transform([clean_article])
    # Keyword frequency
    def extract_keyword_features(texts, keywords):
        features = []
        for text in texts:
            words = text.split()
            counts = [words.count(kw) for kw in keywords]
            features.append(counts)
        return np.array(features)

    keyword_feat = extract_keyword_features([clean_article], keywords)

    # Combine features
    from scipy.sparse import hstack
    combined_feat = hstack([bow_feat, tfidf_feat, keyword_feat])

    # Predict
    prediction = model.predict(combined_feat)

    return prediction[0]

# -----------------------------
# Example usage:
# -----------------------------
new_article = '''The Spaniard capped a remarkable comeback from injury last year, winning his first championship since 2019 after a gruelling struggle that almost forced him to hang up his helmet. The 33-year-old Ducati rider was in imperious form all season, winning 25 races including sprints and clinching the title with five events to spare. Marquez missed the last four races of the season after injuring his collarbone at the Indonesia grand prix with the title already in the bag. Now he is ready to go again, with his sights firmly set on equalling Italian legend Giacomo Agostini's record eight premier class world titles. "We must fight for the title one more time," Marquez told reporters in pre-season testing in Malaysia. "We have the pressure. I mean, if you're in the Ducati Lenovo team then you must fight for the championship, and that is what we will try to do." Marquez spent the winter rehabbing from his latest injury and he admitted that he was not quite back to full fitness in Malaysia. He still put in an impressive performance, although it was his brother Alex Marquez who clocked the fastest times in Sepang.'''
predicted_label = predict_news_category(new_article, lr_model, bow_vectorizer, tfidf_vectorizer, keywords)
print(f"Predicted News Category: {predicted_label}")

## Practice Problems

1. Write two or three short paragraphs of your own (about any topic — a recipe, a movie review, tech news) and run them through `predict_news_category`. Does the model classify them correctly?
2. The keyword list in this notebook was hand-picked. Add 3–5 new keywords of your own to one category and re-run the keyword-related cells. Does it change the `Keyword feature shape` or the per-category top-10 charts?
3. Compare the `accuracy_scores` bar chart against each model's confusion matrix. Pick the model with the lowest accuracy and look at its confusion matrix — which two categories does it confuse most often, and why might that be (hint: think about topic overlap, e.g. business and tech)?
4. Retrain the models using *only* the TF-IDF features (drop BoW and keywords from `X_train_combined`/`X_test_combined`). Does accuracy go up, down, or stay about the same compared to using all three feature types together?
5. Try increasing `max_features` in `CountVectorizer`/`TfidfVectorizer` from 5000 to a larger or smaller value. How does that affect the final feature shape and each model's accuracy?